# PCM Fase 3b — Fine-tune Granite con Unsloth (QLoRA)

Ejecutar en RunPod GPU (RTX 4090). Dataset en `/workspace/train.jsonl` y `/workspace/valid.jsonl`.

In [ ]:
# Instalar dependencias (ejecutar UNA vez; tarda 5-10 min)
import subprocess
import sys

pkgs = [
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "transformers>=4.43.0,<5.0",
    "peft>=0.10.0",
    "datasets>=2.18.0",
    "trl>=0.8.0",
    "accelerate>=0.27.0",
    "bitsandbytes>=0.42.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencias instaladas. Ejecuta la siguiente celda.")


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"  # Verificar equivalencia con granite4.1:3b Ollama
MAX_SEQ_LENGTH = 2048
OUTPUT_DIR = "/workspace/granite-lora"

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing=True,
)

In [ ]:
def format_chat(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = load_dataset("json", data_files="/workspace/train.jsonl", split="train")
valid_ds = load_dataset("json", data_files="/workspace/valid.jsonl", split="train")
train_ds = train_ds.map(format_chat)
valid_ds = valid_ds.map(format_chat)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="no",       # evita checkpoint intermedio (PicklingError TRL)
        save_strategy="no",       # guardamos solo al final con save_pretrained
        report_to="none",
        output_dir=OUTPUT_DIR,
    ),
)

trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter guardado en {OUTPUT_DIR}")
